In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','pandas==2.2.3','numpy==2.3.5','pyarrow==26.0.0','holidays==0.106','catboost==1.2.10','lightgbm==4.7.0','xgboost==3.1.3'])


In [ ]:
from pathlib import Path
p=Path('/kaggle/working/cpu_stage2.py')
p.write_text('"""Kaggle CPU worker reuses private GPU-prepared folds; reserved months stay closed."""\nimport base64,gc,hashlib,io,json,os,sys,zipfile\nfrom pathlib import Path\nimport pandas as pd\nPAYLOAD=\'...\'\nTASK=\'development\'\n\ndef main():\n    root=Path(\'/tmp/x5_cpu_stage2\');parent=root/\'Forecasting\';here=parent/\'research_stage2\';here.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:\n        for name in [\'features.py\',\'geo_control.py\',\'experiment.py\',\'cpu_driver.py\',\'selection.py\']:(here/name).write_bytes(archive.read(name))\n        for name in [\'notebook_api.py\',\'run_validation.py\']:(parent/name).write_bytes(archive.read(name))\n        stores=pd.read_parquet(io.BytesIO(archive.read(\'stores.parquet\')))\n        locations=pd.read_parquet(io.BytesIO(archive.read(\'locations.parquet\')))\n        old_manifest=json.loads(archive.read(\'old_manifest.json\'));expected=json.loads(archive.read(\'expected_legacy.json\'))\n        legacy=archive.read(\'legacy_source.py\').decode()\n    source=list(Path(\'/kaggle/input\').rglob(\'input_manifest.json\'))[0].parent\n    data=list(Path(\'/kaggle/input\').rglob(\'research_manifest.json\'))[0].parent\n    prepared=list(Path(\'/kaggle/input\').rglob(\'prepared_stage2.zip\'))\n    assert len(prepared)==1,\'Attach the completed private phase1 kernel output\'\n    raw_dir=root/\'Data/Shops\';raw_dir.mkdir(parents=True,exist_ok=True)\n    baseline=parent/\'baseline\';baseline.mkdir(exist_ok=True)\n    (raw_dir/\'train_2.csv\').symlink_to(source/\'train_2.csv\');(baseline/\'original_solution.ipynb\').symlink_to(source/\'original_solution.ipynb\')\n    sys.path.insert(0,str(parent));sys.path.insert(0,str(here))\n    from notebook_api import load_recipe,read_raw\n    from experiment import SCREEN,execute\n    from selection import pick_one\n    cache=parent/\'cache\';cache.mkdir(exist_ok=True)\n    new_cache=here/\'cache\';new_cache.mkdir(exist_ok=True)\n    with zipfile.ZipFile(prepared[0]) as archive:\n        assert archive.testzip() is None\n        for item in archive.infolist():\n            if item.filename==\'stage2_features.parquet\':target=new_cache/\'features.parquet\'\n            else:target=cache/item.filename\n            assert target.resolve().is_relative_to(root)\n            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(archive.read(item))\n    # Manifest and causal checks are recomputed using the unchanged builder.\n    from features import compute\n    raw=read_raw(load_recipe());new,info=compute(raw,stores,locations)\n    pd.testing.assert_frame_equal(new,pd.read_parquet(new_cache/\'features.parquet\'),check_exact=True)\n    digest=hashlib.sha256((here/\'features.py\').read_bytes()).hexdigest();info[\'implementation_sha256\']=digest\n    (new_cache/\'feature_manifest.json\').write_text(json.dumps(info,indent=2))\n    (here/\'feature_checks.json\').write_text(json.dumps(dict(status=\'passed\',implementation_sha256=digest,\n        private_phase1_feature_byte_parity=True,outer_targets_never_used_for_stopping=True),indent=2))\n    import geo_control\n    geo=geo_control.compute(raw,stores,locations)\n    geo.to_parquet(new_cache/\'geo_control.parquet\',index=False)\n    (here/\'geo_control_checks.json\').write_text(json.dumps(dict(status=\'passed\',\n        implementation_sha256=hashlib.sha256((here/\'geo_control.py\').read_bytes()).hexdigest(),\n        static_distance_only=True,target_cutoff=\'strictly earlier outcomes\'),indent=2))\n    old_dir=parent/\'research/cache\';old_dir.mkdir(parents=True,exist_ok=True)\n    groups=old_manifest[\'groups\'];columns=list(dict.fromkeys(c for g in groups.values() for c in g))\n    old=pd.concat([pd.read_parquet(data/name,columns=[\'new_id\',\'year_month_index\']+columns)\n        for name in [\'2025-02_train.parquet\',\'2025-02_holdout.parquet\']],ignore_index=True)\n    old.to_parquet(old_dir/\'features.parquet\',index=False);stores.to_parquet(old_dir/\'stores.parquet\',index=False)\n    (old_dir/\'feature_manifest.json\').write_text(json.dumps(old_manifest,indent=2))\n    namespace=dict(STORES=stores,LOCATIONS=locations)\n    exec(compile(legacy,\'preserved_spatial_functions\',\'exec\'),namespace)\n    original,_=namespace[\'spatial_table\'](raw,kind=\'exp50\')\n    fingerprint=hashlib.sha256(pd.util.hash_pandas_object(original[expected[\'columns\']],index=False).values.tobytes()).hexdigest()\n    assert fingerprint==expected[\'sha256\'],\'Original CatBoost spatial builder changed\'\n    original.to_parquet(cache/\'spatial_exp50.parquet\',index=False)\n    del raw,original,old,new;gc.collect()\n    threads=min(8,os.cpu_count() or 4);out=Path(\'/kaggle/working\')\n    def export():\n        with zipfile.ZipFile(out/(\'cpu_development.zip\' if TASK==\'development\' else \'full_august.zip\'),\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n            for folder in [here/\'results\',here/\'inner\']:\n                for p in folder.rglob(\'*\'):\n                    if p.is_file():archive.write(p,p.relative_to(here).as_posix())\n            for p in here.glob(\'confirm_plan_*.json\'):archive.write(p,p.name)\n    try:\n        model=\'catboost_log_level\'\n        if TASK==\'full_august\':\n            for config in [\'baseline\',\'previous\']:execute(model,\'2024-08\',config,\'full\',threads=threads)\n        else:\n            for config in SCREEN:execute(model,\'2024-08\',config,threads=threads)\n            export();configs,scores=pick_one(model)\n            if \'geo8_control\' in configs:execute(model,\'2024-08\',\'geo8_control\',threads=threads)\n            plan=dict(model=model,screen=\'2024-08\',confirm_months=[\'2024-06\',\'2024-07\'],configs=configs,scores=scores)\n            (here/(\'confirm_plan_\'+model+\'.json\')).write_text(json.dumps(plan,indent=2))\n            for fold in [\'2024-06\',\'2024-07\']:\n                for config in configs:execute(model,fold,config,threads=threads)\n                export()\n        (out/\'cpu_checks.json\').write_text(json.dumps(dict(status=\'passed\',backend=\'CPU\',threads=threads,\n            original_legacy_byte_parity=True,legacy_fingerprint=fingerprint,development_only=True,task=TASK,\n            reserved_months_not_evaluated=[\'2024-11\',\'2024-12\']),indent=2))\n    finally:export()\n    print(\'CPU_DEVELOPMENT_DONE\',flush=True)\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
subprocess.check_call([sys.executable,'-u',str(p)])
